# Youtube Phrase (N-grams) Extraction Tool
This notebook contains the implementation for extracting transcripts and analyzing frequent phrases in Youtube videos.

In [1]:
!pip install youtube-transcript-api nltk


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import json
import queue
import re
import concurrent.futures
import os
import time
import random
import requests
from youtube_transcript_api import YouTubeTranscriptApi
from collections import defaultdict, Counter
import nltk
from nltk.corpus import stopwords
import string

# Download the stopwords if they are not already available
nltk.download('stopwords')
stop_words = set(stopwords.words('spanish')) # Assuming Spanish, but it can be modified

def get_free_proxies():
    """Obtiene una lista de proxies gratuitos de free-proxy-list.net"""
    try:
        # Usamos un header de usuario para evitar bloqueos en la obtención de proxies
        headers = {'User-Agent': 'Mozilla/5.0'}
        response = requests.get('https://free-proxy-list.net/', headers=headers, timeout=10)
        proxies = re.findall(r'\d+\.\d+\.\d+\.\d+:\d+', response.text)
        return proxies
    except Exception as e:
        print(f"Error obteniendo proxies: {e}")
        return []

# Configuration
INTERSECTION_THRESHOLD = 0.6

# Directory structure setup
directories_to_create = [
    "processed_files",
    "processed_files/processed_transcriptions",
    "processed_files/processed_n_gramas/creators",
    "processed_files/processed_n_gramas/global/first_appearance",
    "processed_files/processed_n_gramas/global/all_appearances"
]
for directory in directories_to_create:
    os.makedirs(directory, exist_ok=True)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\josel\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Stage 1: Transcripts Download (In Parallel)

In this stage, transcripts are obtained from the provided URLs using `youtube-transcript-api` and saved in temporary JSON files. The identifiers of these files are queued for later processing.

In [3]:
# Native Python queue to pass information to stage 2
processing_queue = queue.Queue()

# 1. Instanciamos el API una sola vez para ser usada globalmente o dentro de las funciones
ytt_api = YouTubeTranscriptApi()

# Native Python queue to pass information to stage 2
processing_queue = queue.Queue()

def extract_video_id(url):
    """Extrae el ID del video de YouTube desde la URL."""
    match = re.search(r'(?:v=|\/)([0-9A-Za-z_-]{11}).*', url)
    return match.group(1) if match else None

def process_video(url, creator):
    """Downloads the video transcript with retries and proxies to avoid IP bans."""
    video_id = extract_video_id(url)
    if not video_id:
        print(f"Error: No se pudo extraer el ID de la URL: {url}")
        return

    max_retries = 5
    base_delay = 5
    
    # Inicializamos lista de proxies en el objeto de la función si no existe
    if not hasattr(process_video, 'proxies_list') or not process_video.proxies_list:
        process_video.proxies_list = get_free_proxies()
        print(f"Proxies cargados: {len(process_video.proxies_list)} disponibles.")

    for attempt in range(max_retries):
        proxy_dict = None
        proxy_used = "None"
        
        if process_video.proxies_list:
            proxy = random.choice(process_video.proxies_list)
            proxy_used = proxy
            proxy_dict = {"http": f"http://{proxy}", "https": f"http://{proxy}"}

        try:
            # IMPORTANTE: No se cambia la lógica de funcionamiento actual, 
            # solo se añade el parámetro opcional de proxy si el método lo soporta.
            try:
                fetched_transcript = ytt_api.fetch(video_id, languages=['es', 'en'], proxies=proxy_dict)
            except TypeError:
                # Si la versión de la librería no acepta proxies en fetch, llamamos normal
                fetched_transcript = ytt_api.fetch(video_id, languages=['es', 'en'])

            # Convertimos el objeto FetchedTranscript a una lista de diccionarios (raw data)
            transcript = fetched_transcript.to_raw_data()

            # Crear carpeta de creador si no existe
            creator_folder = os.path.join("processed_files", "processed_transcriptions", creator)
            os.makedirs(creator_folder, exist_ok=True)

            output_file = os.path.join(creator_folder, f"{video_id}.json")

            # Guardar en JSON en el directorio del creador
            data = {
                "creator": creator,
                "url": url,
                "video_id": video_id,
                "transcript": transcript
            }

            with open(output_file, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            print(f"Success: Transcript of '{creator}' saved (Attempt {attempt+1}, Proxy: {proxy_used})")
            return

        except Exception as e:
            delay = base_delay * (attempt + 1) + random.uniform(2, 5)
            if attempt < max_retries - 1:
                print(f"Attempt {attempt+1} failed for {video_id} (Proxy: {proxy_used}). Retrying in {delay:.2f}s... Error: {str(e)[:100]}")
                # Si el proxy falló, lo eliminamos de la lista para no repetir
                if proxy_used != "None" and proxy_used in process_video.proxies_list:
                    process_video.proxies_list.remove(proxy_used)
                time.sleep(delay)
            else:
                print(f"Error final processing {url} (Creator: {creator}) after {max_retries} attempts: {str(e)}")
def execute_stage_1(url_dictionary, num_threads=4):
    """Ejecuta la descarga de transcripciones en paralelo y encola los creadores."""
    print(f"Starting Stage 1 with {num_threads} threads...")

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Send tasks al pool de hilos
        futures = [executor.submit(process_video, url, creator) for url, creator in url_dictionary.items()]

        # Esperar a que terminen
        concurrent.futures.wait(futures)

    # Enviar creadores unicos a la cola para la etapa 2
    unique_creators = set(url_dictionary.values())
    for creator in unique_creators:
        processing_queue.put(creator)

    print(f"Stage 1 finished. Creators in queue: {processing_queue.qsize()}")

# Ejemplo de uso
test_dictionary = {
    "https://www.youtube.com/watch?v=mJcJ0ikucOY": "XOCAS",
    "https://www.youtube.com/watch?v=aMcTiuqqJas": "GUSGRI",
    "https://www.youtube.com/watch?v=kewy2RUYb3c": "XOCAS",
    "https://www.youtube.com/watch?v=n9RhsFlKeSI": "XOCAS",
    "https://www.youtube.com/watch?v=S8q9eHBnHxE": "XOCAS",
    "https://www.youtube.com/watch?v=dnWT6lV8Zms": "GUSGRI",
    "https://www.youtube.com/watch?v=yUAbe5isKTg": "GUSGRI",
    "https://www.youtube.com/watch?v=5rF-0wK4fpI": "GUSGRI",
    "https://www.youtube.com/watch?v=R2BYWHmhBuk": "GUSGRI",
    "https://www.youtube.com/watch?v=8V6b5hCx5F8": "XOCAS",
    "https://www.youtube.com/watch?v=tf4LAo_JY68": "XOCAS",
    "https://www.youtube.com/watch?v=Bfy1yDB_YXM": "XOCAS",
    "https://www.youtube.com/watch?v=JaqdFMzWmqU": "XOCAS",
    "https://www.youtube.com/watch?v=tUsXRMXlHl0": "GUSGRI",
    "https://www.youtube.com/watch?v=PYPsdJmcgjo": "GUSGRI",
    "https://www.youtube.com/watch?v=i49LvJMtwbM": "GUSGRI",
    "https://www.youtube.com/watch?v=p5jo4kS2P8Q": "GUSGRI",
    "https://www.youtube.com/watch?v=fQxVMOSB_L8": "GUSGRI",
    "https://www.youtube.com/watch?v=SOnBk94vOzk": "GUSGRI",
    "https://www.youtube.com/watch?v=-ibeiAm_VEY": "GUSGRI"
}

# Execute Stage 1
execute_stage_1(test_dictionary, num_threads=2)


Starting Stage 1 with 2 threads...
Proxies cargados: 300 disponibles.
Proxies cargados: 300 disponibles.
Success: Transcript of 'GUSGRI' saved (Attempt 1, Proxy: 46.249.100.124:80)
Success: Transcript of 'XOCAS' saved (Attempt 1, Proxy: 157.245.40.210:80)
Success: Transcript of 'XOCAS' saved (Attempt 1, Proxy: 107.173.150.106:6560)
Success: Transcript of 'XOCAS' saved (Attempt 1, Proxy: 165.101.102.206:8088)
Success: Transcript of 'XOCAS' saved (Attempt 1, Proxy: 47.250.177.202:20000)
Success: Transcript of 'GUSGRI' saved (Attempt 1, Proxy: 209.50.162.57:3129)
Success: Transcript of 'GUSGRI' saved (Attempt 1, Proxy: 64.188.77.221:3128)
Success: Transcript of 'GUSGRI' saved (Attempt 1, Proxy: 216.173.120.150:6442)
Success: Transcript of 'GUSGRI' saved (Attempt 1, Proxy: 146.103.44.46:6598)
Attempt 1 failed for tf4LAo_JY68 (Proxy: 198.46.137.141:6345). Retrying in 8.76s... Error: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=tf4LAo_JY68! This is 
Success:

## Stage 2: N-grams Processing (In Parallel)

In this stage, the queued files from Stage 1 are processed. The most used n-grams by each creator are extracted (from 2 to 6 words), optionally discarding those that are composed 100% of stop words. The timestamp of the first appearance of the n-gram is captured.

In [4]:
def clean_word(word):
    """Converts to lowercase and removes punctuation signs."""
    word = word.lower()
    return word.translate(str.maketrans('', '', string.punctuation))

def is_valid_ngram(ngram_words, use_stopwords_filter=True):
    """
    Business rule: If the filter is active, the n-gram is discarded
    SOLO si TODAS las words que lo componen son stopwords.
    """
    if not use_stopwords_filter:
        return True

    # Validate if all words are in the stopwords list
    are_all_stopwords = all(word in stop_words for word in ngram_words)

    # It is valid if NOT all are stopwords (i.e., it has at least one valuable word)
    return not are_all_stopwords

def tokenize_transcript(transcript):
    """
    Decomposes the transcript into a list of words where each word
    retiene el inicio (start) y el fin (start + duration) de su block original.
    """
    tokens = []
    for block in transcript:
        text = block.get('text', '')
        start = block.get('start', 0)
        end = start + block.get('duration', 0)

        words = text.split()
        for p in words:
            clean_p = clean_word(p)
            if clean_p: # Avoid empty strings
                tokens.append({
                    'word': clean_p,
                    'start': start,
                    'end': end
                })
    return tokens

def extract_creator_ngrams(creator, n_phrases, use_stopwords_filter=True):
    """Processes all JSON files for a creator, generates n-grams and separates union/intersection."""
    creator_folder = os.path.join("processed_files", "processed_transcriptions", creator)
    if not os.path.exists(creator_folder):
        print(f"Folder not found for creator {creator}")
        return None, None

    json_files = [f for f in os.listdir(creator_folder) if f.endswith('.json')]
    total_videos = len(json_files)
    if total_videos == 0:
        return None, None

    # { n_size: { "ngrama_str": {"count": int, "video_appearances": set(video_id), "start": float, "end": float, "url": str, "appearances": []} } }
    creator_results = {i: {} for i in range(2, 7)}

    for file_name in json_files:
        file_path = os.path.join(creator_folder, file_name)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)

            transcript = data['transcript']
            url = data['url']
            video_id = data['video_id']

            tokens = tokenize_transcript(transcript)
            total_tokens = len(tokens)

            for n in range(2, 7):
                for i in range(total_tokens - n + 1):
                    ngram_tokens = tokens[i:i+n]
                    ngram_words = tuple(t['word'] for t in ngram_tokens)

                    if not is_valid_ngram(ngram_words, use_stopwords_filter):
                        continue

                    ngram_str = " ".join(ngram_words)
                    start_time = ngram_tokens[0]['start']
                    end_time = ngram_tokens[-1]['end']
                    appearance = {'start': start_time, 'end': end_time, 'url': url}

                    if ngram_str in creator_results[n]:
                        creator_results[n][ngram_str]['count'] += 1
                        creator_results[n][ngram_str]['video_appearances'].add(video_id)
                        creator_results[n][ngram_str]['appearances'].append(appearance)
                    else:
                        creator_results[n][ngram_str] = {
                            'count': 1,
                            'video_appearances': {video_id},
                            'start': start_time,
                            'end': end_time,
                            'url': url,
                            'appearances': [appearance]
                        }
        except Exception as e:
            print(f"Error processing file {file_path}: {str(e)}")

    # Separate into union and intersection
    min_videos_required = total_videos * INTERSECTION_THRESHOLD
    
    union_results = {i: [] for i in range(2, 7)}
    intersection_results = {i: [] for i in range(2, 7)}

    for n in range(2, 7):
        all_ngrams = []
        intersect_ngrams = []
        
        for ngram_str, info in creator_results[n].items():
            # Remove set before returning/sorting (sets are not JSON serializable)
            videos_count = len(info['video_appearances'])
            clean_info = {k: v for k, v in info.items() if k != 'video_appearances'}
            
            all_ngrams.append((ngram_str, clean_info))
            if videos_count >= min_videos_required:
                intersect_ngrams.append((ngram_str, clean_info))

        # Sort and take top N
        all_ngrams.sort(key=lambda x: x[1]['count'], reverse=True)
        intersect_ngrams.sort(key=lambda x: x[1]['count'], reverse=True)
        
        union_results[n] = all_ngrams[:n_phrases]
        intersection_results[n] = intersect_ngrams[:n_phrases]

    return creator, {'union': union_results, 'intersection': intersection_results}

def execute_stage_2(n_phrases=10, num_threads=4, use_stopwords_filter=True):
    """Empties the queue, processes in parallel by creator, and extracts the Top N phrases."""
    creators_to_process = []

    # Empty the queue
    while not processing_queue.empty():
        creators_to_process.append(processing_queue.get())

    if not creators_to_process:
        print("The queue is empty. Execute Stage 1 first.")
        return None

    print(f"Starting Stage 2: Processing {len(creators_to_process)} creators with {num_threads} threads...")

    # Structure: { creator: {'union': {n: [(phrase, info)]}, 'intersection': {n: [(phrase, info)]}} }
    top_results = {}

    with concurrent.futures.ThreadPoolExecutor(max_workers=num_threads) as executor:
        # Send tasks
        futures = [executor.submit(extract_creator_ngrams, creator, n_phrases, use_stopwords_filter) for creator in creators_to_process]

        # Collect results as they finish
        for futuro in concurrent.futures.as_completed(futures):
            creator, creator_results = futuro.result()
            if creator and creator_results:
                top_results[creator] = creator_results

    print("Stage 2 finished.")
    return top_results

def format_results(creator_results, mode):
    """Formats the ngrams results into either 'first_appearance' or 'all_appearances' format."""
    formatted = {}
    for n in range(2, 7):
        formatted[f"{n}_grams"] = []
        for phrase, info in creator_results.get(n, []):
            count = info['count']
            if mode == 'first_appearance':
                formatted[f"{n}_grams"].append({
                    'phrase': phrase,
                    'repetitions': count,
                    'first_appearance': {'start': info['start'], 'end': info['end'], 'url': info['url']}
                })
            else:
                formatted[f"{n}_grams"].append({
                    'phrase': phrase,
                    'repetitions': count,
                    'appearances': info.get('appearances', [])
                })
    return formatted

def show_results_and_save(top_results):
    """Formats, displays and saves the results in the correct directory structures."""
    if not top_results:
        return

    # Global results holders
    global_union_first = {}
    global_union_all = {}
    global_interception_first = {}
    global_interception_all = {}

    for creator, results in top_results.items():
        # Format creator results for both modes and both metrics
        union_first = format_results(results['union'], 'first_appearance')
        union_all = format_results(results['union'], 'all_appearances')
        interception_first = format_results(results['intersection'], 'first_appearance')
        interception_all = format_results(results['intersection'], 'all_appearances')

        # Save global data
        global_union_first[creator] = union_first
        global_union_all[creator] = union_all
        global_interception_first[creator] = interception_first
        global_interception_all[creator] = interception_all

        # Ensure creator specific directories
        creator_first_dir = os.path.join("processed_files", "processed_n_gramas", "creators", creator, "first_appearance")
        creator_all_dir = os.path.join("processed_files", "processed_n_gramas", "creators", creator, "all_appearances")
        os.makedirs(creator_first_dir, exist_ok=True)
        os.makedirs(creator_all_dir, exist_ok=True)

        # Save creator files
        with open(os.path.join(creator_first_dir, "n_gramas_union.json"), 'w', encoding='utf-8') as f:
            json.dump(union_first, f, ensure_ascii=False, indent=2)
        with open(os.path.join(creator_first_dir, "n_gramas_interception.json"), 'w', encoding='utf-8') as f:
            json.dump(interception_first, f, ensure_ascii=False, indent=2)
        with open(os.path.join(creator_all_dir, "n_gramas_union.json"), 'w', encoding='utf-8') as f:
            json.dump(union_all, f, ensure_ascii=False, indent=2)
        with open(os.path.join(creator_all_dir, "n_gramas_interception.json"), 'w', encoding='utf-8') as f:
            json.dump(interception_all, f, ensure_ascii=False, indent=2)

        print(f"\n{'='*50}")
        print(f"Saved files for CREATOR: {creator}")
        print(f"{'='*50}")

    # Save global files
    global_first_dir = os.path.join("processed_files", "processed_n_gramas", "global", "first_appearance")
    global_all_dir = os.path.join("processed_files", "processed_n_gramas", "global", "all_appearances")

    with open(os.path.join(global_first_dir, "n_gramas_union.json"), 'w', encoding='utf-8') as f:
        json.dump(global_union_first, f, ensure_ascii=False, indent=2)
    with open(os.path.join(global_first_dir, "n_gramas_interception.json"), 'w', encoding='utf-8') as f:
        json.dump(global_interception_first, f, ensure_ascii=False, indent=2)
    with open(os.path.join(global_all_dir, "n_gramas_union.json"), 'w', encoding='utf-8') as f:
        json.dump(global_union_all, f, ensure_ascii=False, indent=2)
    with open(os.path.join(global_all_dir, "n_gramas_interception.json"), 'w', encoding='utf-8') as f:
        json.dump(global_interception_all, f, ensure_ascii=False, indent=2)

    print("\nAll global results successfully generated and saved.")

# Execute Stage 2 and show results
results = execute_stage_2(n_phrases=10, num_threads=2, use_stopwords_filter=True)
show_results_and_save(results)


Starting Stage 2: Processing 2 creators with 2 threads...
Stage 2 finished.

Saved files for CREATOR: GUSGRI

Saved files for CREATOR: XOCAS

All global results successfully generated and saved.
